# Notebook 13 — Genome Browsers, IGV, and Variant Visualization

**Module:** 09 — Genomics and NGS  
**Notebook:** 13 of 16  
**Time estimate:** 60 minutes

> Reading IGV screenshots is a core interview skill. You must be able to identify
> SNPs, indels, duplicates, and strand bias from a coverage track image.

---
## Step 1 — Motivation

Numbers alone (VAF, QUAL, MAPQ) don't tell the whole story. IGV (Integrative
Genomics Viewer) lets you look directly at the aligned reads at a variant site.
Many false positives and real variants can only be distinguished visually — soft
clips concentrated at an artifact, strand bias visible as color patterns, or
a clean het SNP with uniform read distribution.

---
## Step 2 — Intuition

**IGV shows (from top to bottom):**
1. **Reference track:** The reference genome sequence
2. **Coverage track:** Grey histogram showing depth at each position
3. **Read track:** Individual aligned reads, colored by strand by default
   (blue = forward strand, red/pink = reverse strand)
4. **Annotation track:** Gene models from GTF (optional)

**What to look for at a variant site:**
- **Real SNP:** ~50% reads show alternate base, evenly distributed between
  forward and reverse strands, high-quality bases (no color changes within reads)
- **Sequencing artefact:** Variant only on one strand (strand bias), or only
  near the end of reads (positional bias), or all in soft-clipped reads
- **Indel:** Reads show soft clips or gaps at the same position; CIGAR 'I' or 'D'
- **Duplicate:** Multiple reads with identical start/end positions (same color,
  same length, same sequence)

---
## Step 3 — Biological Background

**UCSC Genome Browser vs IGV:**

| Feature | IGV | UCSC Genome Browser |
|---------|-----|--------------------|
| Local vs. web | Local desktop app | Web browser |
| Custom tracks | Load local BAM/VCF | Upload to UCSC |
| Read-level view | Yes | Limited |
| Population data | No | Yes (gnomAD, GTEx) |
| Primary use | Inspecting your data | Exploring public data |

**Strand bias artefacts:**  
Some PCR-based library preparation introduces systematic errors on one strand.
DNA oxidation (8-oxoguanine) causes G→T transversions predominantly on one strand.
FFPE (formalin-fixed paraffin-embedded) samples have deamination of cytosine → uracil,
causing C→T transitions. Both produce strong strand bias in IGV.

In [ ]:
# Step 6 — Python: Simulate an IGV-like read visualization
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass


@dataclass
class SimRead:
    start: int      # genomic start (0-based)
    length: int
    strand: str     # '+' or '-'
    bases: str
    mapq: int
    is_duplicate: bool = False


def simulate_igv_pileup(
    ref: str,
    snp_pos: int,
    snp_alt: str,
    n_reads: int = 40,
    vaf: float = 0.5,
    strand_bias: float = 0.0,  # 0=balanced, 1=all ALT on forward strand
    dup_fraction: float = 0.1,
    read_len: int = 50,
    seed: int = 42
) -> list[SimRead]:
    rng = np.random.default_rng(seed)
    region_start = max(0, snp_pos - read_len // 2)
    region_end = min(len(ref), snp_pos + read_len // 2)
    reads = []

    for i in range(n_reads):
        start = int(rng.integers(region_start, max(region_start + 1, region_end - read_len + 1)))
        end = min(start + read_len, len(ref))
        bases = list(ref[start:end])

        is_alt = rng.random() < vaf
        strand = '+' if rng.random() < 0.5 else '-'

        # Strand bias: alt reads preferentially on forward strand
        if is_alt and strand_bias > 0:
            strand = '+' if rng.random() < (0.5 + strand_bias * 0.5) else '-'

        if is_alt and start <= snp_pos < end:
            bases[snp_pos - start] = snp_alt

        # Add errors
        for j in range(len(bases)):
            if rng.random() < 0.005:
                bases[j] = rng.choice([b for b in 'ACGT' if b != bases[j]])

        is_dup = rng.random() < dup_fraction
        reads.append(SimRead(
            start=start, length=end-start, strand=strand,
            bases=''.join(bases), mapq=int(rng.integers(30, 61)),
            is_duplicate=is_dup
        ))

    # Sort by start position
    return sorted(reads, key=lambda r: r.start)


def plot_igv_view(
    ax, reads: list[SimRead], ref: str, snp_pos: int, snp_alt: str,
    view_start: int, view_end: int, title: str
):
    """
    Draw an IGV-style pileup visualization.
    """
    n_pos = view_end - view_start

    # Coverage
    cov = np.zeros(n_pos)
    for r in reads:
        for j in range(r.length):
            pos = r.start + j - view_start
            if 0 <= pos < n_pos:
                cov[pos] += 1
    # Draw coverage histogram at top
    cov_height = 0.25  # fraction of axes height
    max_cov = max(cov.max(), 1)
    cov_y = [1 - cov_height + (c / max_cov) * cov_height for c in cov]

    # Assign reads to rows (greedy packing)
    rows = []
    read_rows = []
    for r in reads:
        placed = False
        for row_idx, row_end in enumerate(rows):
            if r.start > row_end:
                rows[row_idx] = r.start + r.length
                read_rows.append(row_idx)
                placed = True
                break
        if not placed:
            rows.append(r.start + r.length)
            read_rows.append(len(rows) - 1)

    n_rows = max(read_rows) + 1 if read_rows else 1
    read_area_height = 1 - cov_height - 0.05
    row_height = read_area_height / max(n_rows, 1)

    ax.set_xlim(view_start, view_end)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Draw coverage
    ax.fill_between(
        range(view_start, view_end),
        [1 - cov_height] * n_pos, cov_y,
        color='lightgray', edgecolor='none'
    )
    ax.axhline(1 - cov_height, color='gray', lw=0.5)

    # Draw reads
    base_colors = {'A': '#4FC3F7', 'C': '#81C784', 'G': '#FFB74D', 'T': '#F06292',
                   '-': 'black'}
    snp_offset = snp_pos - view_start

    for r, row_idx in zip(reads, read_rows):
        y_top = (1 - cov_height - 0.03) - row_idx * row_height
        y_bot = y_top - row_height * 0.8
        color = '#5C97D8' if r.strand == '+' else '#E07060'
        if r.is_duplicate:
            color = '#B0B0B0'

        ax.add_patch(mpatches.Rectangle(
            (r.start, y_bot), r.length, y_top - y_bot,
            facecolor=color, edgecolor='none', alpha=0.7
        ))

        # Highlight mismatches (SNP bases)
        for j, b in enumerate(r.bases):
            pos = r.start + j
            ref_b = ref[pos] if pos < len(ref) else 'N'
            if b != ref_b and view_start <= pos < view_end:
                ax.add_patch(mpatches.Rectangle(
                    (pos, y_bot), 1, y_top - y_bot,
                    facecolor=base_colors.get(b, 'white'), edgecolor='none', alpha=0.9
                ))

    # Mark SNP position
    ax.axvline(snp_pos + 0.5, color='red', lw=1, ls='--', alpha=0.7)
    ax.set_title(title, fontsize=9, fontweight='bold')


# Test reference and scenarios
ref = 'ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG'
snp_pos = 35
snp_alt = 'T'  # ref is 'G'

scenarios = [
    ('Real het SNP (no bias)', dict(vaf=0.5, strand_bias=0.0, dup_fraction=0.0)),
    ('Strand-biased artefact', dict(vaf=0.3, strand_bias=0.9, dup_fraction=0.0)),
    ('High duplicate rate', dict(vaf=0.5, strand_bias=0.0, dup_fraction=0.4)),
    ('Low-VAF variant (somatic?)', dict(vaf=0.12, strand_bias=0.0, dup_fraction=0.0)),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (title, kwargs) in zip(axes.flatten(), scenarios):
    reads = simulate_igv_pileup(ref, snp_pos, snp_alt, n_reads=50, **kwargs)
    plot_igv_view(ax, reads, ref, snp_pos, snp_alt,
                  view_start=15, view_end=65, title=title)
    # Add legend
    alt_reads = [r for r in reads if snp_pos >= r.start and snp_pos < r.start + r.length
                 and r.bases[snp_pos - r.start] == snp_alt]
    all_covering = [r for r in reads if snp_pos >= r.start and snp_pos < r.start + r.length]
    vaf_obs = len(alt_reads) / len(all_covering) if all_covering else 0
    ax.text(0.02, 0.05, f'VAF={vaf_obs:.2f}, n_reads={len(reads)}',
            transform=ax.transAxes, fontsize=8)

plt.suptitle('IGV-style Read Pileup Visualization\n'
             '(Blue=forward strand, Red=reverse strand, Colored=mismatch/SNP)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('igv_pileup.png', dpi=150, bbox_inches='tight')
plt.show()

---
## IGV inspection checklist (for every variant you examine)

| Check | Good sign | Warning sign |
|-------|-----------|-------------|
| Strand balance | ~50:50 fwd/rev for ALT reads | >80% ALT on one strand |
| Read position | ALT bases spread across read positions | All ALT near read ends |
| Soft clips | Rare | Many reads clipped at same position |
| Duplicate rate | <20% | Stacks of identical reads |
| MAPQ | Most reads MAPQ≥20 | Dark/grey reads (low MAPQ) |
| Coverage | Uniform around variant | Drop-offs adjacent to variant |
| VAF | ~50% (het) or ~100% (hom) for germline | <10% unexpected in germline |

---
## Step 8 — Exercises

1. Describe what an FFPE artefact looks like in IGV (hint: C→T transitions, one strand).
2. How do you distinguish a real 3 bp deletion from reads that are soft-clipped at
   the same position due to an alignment artefact?
3. In IGV, you see a putative SNP with VAF=0.15 in a WGS sample. Half the reads
   supporting the ALT allele are in soft-clipped portions. Is this a real variant?

---
## Step 10 — Quiz

1. What do blue and red/pink reads represent in IGV's default color scheme?
2. How would a PCR duplicate look in an IGV pileup?
3. What is strand bias and how can you detect it visually in IGV?
4. A variant has 30× coverage but only 2 ALT reads, both on the forward strand.
   What does this suggest?
5. Name three types of artefacts you can identify in IGV that a variant caller
   might still report as PASS.

---
## Step 12 — Reflection

> *[In one paragraph: describe a situation where a variant calling pipeline
> reports a PASS SNP, but after visually inspecting it in IGV you would decide
> NOT to follow it up for clinical or experimental validation.]*

---
*Next: `14_mini_project_variant_calling_pipeline.ipynb`*